# The DS inner loop on the Snowflake plane

The same wide batch-monthly churn cadence as `ds_inner_loop.ipynb`, reading **Snowflake** instead of the SeaweedFS lake.
Not a second project and not a second set of specs: every table in `sources.yml` declares an `identifier:` beside its `path:`, each adapter reads only the field it understands, and the dataset/model/scoring specs are untouched.
Switching the data plane really is one word, and this notebook is that word plus the exploration around it.

**This notebook does not run in the showcase's JupyterLab at http://localhost:8899.**
That kernel is the runner image, which deliberately ships no `mbt-snowflake` (its `mbt-h2o[sparkling]` extra pins pyspark 3.5.x, which does not resolve against the connector's `cryptography` floor), and `externalbrowser` SSO needs a real browser with a localhost callback.
Launch a kernel from the repo's own environment on your host instead:

```bash
cd <repo root>
set -a; source examples/showcase/.env; set +a
SHOWCASE_MLFLOW_URI=http://localhost:5501 \
AWS_ACCESS_KEY_ID=mbtadmin AWS_SECRET_ACCESS_KEY=mbtsecret \
AWS_ENDPOINT_URL_S3=http://localhost:8333 AWS_DEFAULT_REGION=us-east-1 \
  uv run --with jupyterlab jupyter lab --notebook-dir examples/showcase/project
```

`make up` still has to be running: this plane reads data from Snowflake, but tracking, the registry, and the artifact store are the compose stack's, reached over published host ports.
And `make snowflake-seed` must have run, or the compile fails before it reaches your selection.

Background: `examples/showcase/README.md` ("The warehouse plane"), `DESIGN.md` section 11, `docs/naming-conventions.md`.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

import pandas as pd

# The kernel may start in notebooks/; every mbt command runs from the project root.
if not Path("mbt_project.yml").exists() and Path("../mbt_project.yml").exists():
    os.chdir("..")
assert Path("mbt_project.yml").exists(), (
    "run this from examples/showcase/project (see the launch command above)"
)

# AWS_* are required even though this plane never touches s3a: profiles.yml is
# rendered WHOLE whichever target you pick, and the shared s3a anchor calls
# env_var('AWS_ACCESS_KEY_ID') with no default. Without them every mbt command
# below dies in profile rendering, which reads as a Snowflake problem and is not.
required = [
    "SNOWFLAKE_ACCOUNT",
    "SNOWFLAKE_USER",
    "SNOWFLAKE_WAREHOUSE",
    "SNOWFLAKE_DATABASE",
    "SNOWFLAKE_SCHEMA",
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
]
missing = [k for k in required if not os.environ.get(k, "").strip()]
assert not missing, f"missing environment: {', '.join(missing)}"

# Trailing whitespace in an identifier is silent until it is not: interpolated
# SQL ignores it, bound identifiers do not (docs/troubleshooting.md).
for key in ("SNOWFLAKE_DATABASE", "SNOWFLAKE_SCHEMA"):
    assert os.environ[key] == os.environ[key].strip(), f"{key} has surrounding whitespace"

# Pinned to the seeded data range, as everywhere else in the showcase. In
# production this is the Airflow logical date (docs/naming-conventions.md).
ANCHOR = "2026-06-30T00:00:00Z"
TARGET = "snowflake"
print("project root:", Path.cwd())
print(f"warehouse:   {os.environ['SNOWFLAKE_DATABASE']}.{os.environ['SNOWFLAKE_SCHEMA']}")

## 1. Explore the raw tables, in the warehouse

The seeder uploaded the *same parquet* the lake is seeded from rather than generating fresh data in-warehouse, which is what makes the two planes comparable.
Every wide table joins on ONE key, `inference_date`; the population spine also carries the entity crosswalk (`customer_id` to `safe_id`) and the informational `as_of_date` / `loaded_at_time` columns.

One case gotcha: the seeder writes with `quote_identifiers=False`, so columns fold to UPPERCASE in Snowflake.
mbt's adapter lowercases them on the way back (`normalize_case`), but a raw connector query does not, so `sf()` below does it for you.

In [ ]:
import snowflake.connector

_conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database=os.environ["SNOWFLAKE_DATABASE"],
    schema=os.environ["SNOWFLAKE_SCHEMA"],
    **(
        {"authenticator": os.environ["SNOWFLAKE_AUTHENTICATOR"]}
        if os.environ.get("SNOWFLAKE_AUTHENTICATOR")
        else {}
    ),
    **(
        {"private_key_file": os.environ["SNOWFLAKE_PRIVATE_KEY_FILE"]}
        if os.environ.get("SNOWFLAKE_PRIVATE_KEY_FILE")
        else {}
    ),
    **(
        {"password": os.environ["SNOWFLAKE_PASSWORD"]}
        if os.environ.get("SNOWFLAKE_PASSWORD")
        else {}
    ),
    **({"role": os.environ["SNOWFLAKE_ROLE"]} if os.environ.get("SNOWFLAKE_ROLE") else {}),
    client_store_temporary_credential=True,  # one SSO prompt, not one per cell
)


def sf(sql: str) -> pd.DataFrame:
    """Run SQL, return a DataFrame with mbt's lowercase column convention."""
    with _conn.cursor() as cur:
        frame = cur.execute(sql).fetch_pandas_all()
    frame.columns = frame.columns.str.lower()
    return frame


sf("""
    SELECT inference_date, COUNT(*) AS customers
    FROM MBT_SHOWCASE_MONTHLY_POPULATION
    GROUP BY inference_date ORDER BY inference_date
""")

In [ ]:
# The label contract: rows appear only once the outcome window has closed, so
# the newest cohort has no labels yet at the build anchor and the plain inner
# join in the dataset spec cannot leak immature outcomes.
labels = sf("""
    SELECT inference_date, COUNT(*) AS labeled, AVG(is_churn::FLOAT) AS churn_rate
    FROM MBT_SHOWCASE_MONTHLY_LABELS
    GROUP BY inference_date ORDER BY inference_date
""")
print(labels.to_string(index=False))

# The crosswalk that makes transactions reachable: they key on safe_id, and
# only the spine knows which customer_id that is.
sf("""
    SELECT customer_id, safe_id, as_of_date, inference_date
    FROM MBT_SHOWCASE_MONTHLY_POPULATION
    WHERE inference_date = (SELECT MAX(inference_date) FROM MBT_SHOWCASE_MONTHLY_POPULATION)
    LIMIT 5
""")

## 2. The model is YAML, and it is the same YAML

You do not define the training set in this notebook - you review the declaration.
Nothing below mentions Snowflake: the dataset spec names sources, and which address those sources resolve to is a property of the target.
That is the whole point of the plane split, so it is worth reading once with that in mind.

In [ ]:
print(Path("datasets/wide_churn_training.yml").read_text())

In [ ]:
# The dual addressing that makes it work. Each adapter reads exactly one of
# these two fields; spark is the only one that could read either, which is why
# the lake targets set `source_address: path` explicitly.
import yaml

tables = yaml.safe_load(Path("sources.yml").read_text())["sources"][0]["tables"]
pd.DataFrame(
    [
        {
            "table": t["name"],
            "path (lake planes)": t["path"],
            "identifier (this plane)": t["identifier"],
        }
        for t in tables
        if t["name"]
        in {
            "monthly_population",
            "monthly_labels",
            "demographic_history",
            "login_history",
            "transaction_history",
            "wide_churn_outcomes",
        }
    ]
)

## 3. The fast inner loop: build the probe on the warehouse

`mbt build` compiles a pinned manifest, pushes the joins and filters down as SQL, streams the result back as Arrow into the standard local materialization, trains, evaluates gates, and writes machine-readable results.
Exit codes mean something: 0 trained and passed, 2 means a gate or check said no (feedback, not breakage), 1 is a real error.

Two things differ from the lake notebook and neither is in the specs.
Snapshot pinning covers **every** source referenced by any node regardless of `--select`, which is why the seeder creates all 12 tables even though this cadence reads 6.
And AutoML trains in a local H2O JVM here (`h2o_backend: local`), because there is no Spark cluster on your host - the lake plane's `prod` target uses sparkling instead.

In [ ]:
!mbt build --target snowflake --select churn_wide_probe --anchor 2026-06-30T00:00:00Z

## 4. Analyze what the run produced

Everything mbt writes is a file this notebook can read: metrics and gates in `target/run_results.json`, and the EXACT frame the model saw under `target/datasets/`.
This is where notebook strengths matter - slice the panel, inspect importances, question the gates.

In [ ]:
results = json.loads(Path("target/run_results.json").read_text())
probe = next(r for r in results["results"] if r["unique_id"].endswith(".churn_wide_probe"))
print("metrics:", {k: round(v, 4) for k, v in probe["metrics"].items()})
print("gates:", probe["gates"])
importance = pd.Series(probe["feature_importance"]).sort_values(ascending=False)
importance.head(10)

In [ ]:
newest = max(
    Path("target/datasets/wide_churn_training").glob("*/train.parquet"),
    key=lambda p: p.stat().st_mtime,
)
train = pd.read_parquet(newest)
print(f"train panel: {train.shape[0]} rows x {train.shape[1]} columns")
train.groupby("inference_date")[["login_days_30d", "txn_cnt_30d", "is_churn"]].mean().round(3)

## 5. Feature selection is a committed diff

`scripts/select_features.py` runs the ds-helper funnel over the materialized train split (drop high-missing, drop single-value, drop correlated pairs, then a seeded LightGBM randomized search keeping importance > 0) and rewrites the include list in `models/churn_wide_automl.yml`.
It honors the model's `exclude:` list - the DS ignored-columns contract - and it is deterministic.

Because the warehouse holds the same bytes the lake does, this reproduces the committed list here too, so the cell leaves no diff.
That is a real cross-plane check, not a formality: a divergence here means the two planes are not seeing the same data.

In [ ]:
!python scripts/select_features.py
!git diff --stat -- models/churn_wide_automl.yml

In [ ]:
report = json.loads(Path("target/feature_selection_report.json").read_text())
stages = report["stages"]
print("candidates:", report["n_candidate_features"], "| excluded by contract:", report["excluded"])
print("high-missing dropped:", len(stages["high_missing"]["dropped"]))
print("single-value dropped:", len(stages["single_unique"]["dropped"]))
print("correlated dropped:", stages["correlated"]["dropped"])
print("zero-importance dropped:", len(stages["lgbm"]["zero_importance_dropped"]))
print("cv roc_auc:", round(stages["lgbm"]["best_cv_roc_auc"], 4))
pd.DataFrame(report["selected"])

## 6. Sample without leaving the warehouse

`sample_fraction` is not a post-hoc filter: it becomes `MOD(MD5_NUMBER_LOWER64(customer_id), 1e6) < fraction * 1e6` inside the pushed-down query, so a 10% slice of a 7M-row table never crosses the wire.
`sample_key: customer_id` hashes the customer, so every snapshot of a kept customer is kept together and smaller fractions are subsets of larger ones.

Point the funnel at a SCRATCH copy so the committed contract stays clean while you compare selections.
(To regenerate the committed list afterwards, rebuild at full fraction first - the funnel always reads the newest complete materialization.)

In [ ]:
import shutil

scratch = Path("/tmp/sampled_experiment_snowflake.yml")
shutil.copy("models/churn_wide_automl.yml", scratch)

subprocess.run(
    [
        "mbt",
        "build",
        "--target",
        TARGET,
        "--select",
        "churn_wide_probe",
        "--anchor",
        ANCHOR,
        "--vars",
        "sample_fraction: 0.25",
    ],
    check=True,
)
sampled_train = max(
    Path("target/datasets/wide_churn_training").glob("*/train.parquet"),
    key=lambda p: p.stat().st_mtime,
)
subprocess.run(
    [
        "python",
        "scripts/select_features.py",
        "--train-parquet",
        str(sampled_train),
        "--model-file",
        str(scratch),
        "--report",
        "target/sampled_selection_report.json",
    ],
    check=True,
)
print("\ncommitted contract untouched:")
!git diff --stat -- models/churn_wide_automl.yml

## 7. Train the serving model, then promote

`churn_wide_automl` trains H2O AutoML on the selected columns.
It registers as **`churn_wide_automl_snowflake`**, not `churn_wide_automl`: `plane_suffix` keeps the two planes' versions from interleaving in the shared registry, where they would quietly corrupt champion resolution.
The node name never changes, so the scoring spec needs no edit.

Rebuild at full fraction first if you ran the sampled experiment above.

In [ ]:
!mbt build --target snowflake --select churn_wide_automl --anchor 2026-06-30T00:00:00Z

In [ ]:
# --target matters here: without it promote falls back to dev, whose registry
# URI is the in-network http://mlflow:5000 that your host cannot reach.
!mbt promote --target snowflake --model churn_wide_automl_snowflake --to production

## 8. Where the notebook ends

Scoring and monitoring are the platform's job, not the notebook's, and they are one command each:

```bash
mbt score   --project-dir . --target snowflake --select tag:wide --anchor 2026-06-30T00:00:00Z
mbt monitor --project-dir . --target snowflake --select tag:wide --anchor 2026-07-20T00:00:00Z
```

`make snowflake` is exactly those two plus the build and promote you just ran.
Predictions stage as parquet under `target/snowflake_predictions/` - ADR-23 v1.
The warehouse-native prediction store is v2 and gated on live verification of the serving leg (issue #1), so predictions do not land in a Snowflake table yet.

Look at the results in MLflow (http://localhost:5501): `churn_wide_automl_snowflake` versions, the `production` alias the promote just moved, per-run metrics, and the `mbt.*` provenance tags.

The model itself stayed in reviewed YAML throughout.
Everything you decided is sitting in files a reviewer can read - the dataset and model specs, the rewritten include and `categorical` lists, and the selection report - so commit them on a branch and open a PR.
From there the platform takes over, and none of it needs this notebook.